In [ ]:
import os 
import certifi 

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI 
from langchain_community.tools.tavily_search import TavilySearchResults  
from langchain import hub 
from langchain.tools import tool
import requests

In [27]:
from langchain.agents import create_react_agent ,AgentExecutor
from langchain.agents import AgentType

In [28]:
# Load environment variables from .env file


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [29]:
search_tool = TavilySearchResults(max_results=5, api_key=TAVILY_API_KEY)

In [30]:
@tool 
def get_weather_data(city : str) -> str:
    """
    Fetch current weather information for a given city 
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" in data:
        return f"Could not fetch weather data for {city}. Please check the city name and try again."

    return (
        
    )

In [31]:
search_tool.invoke("Give me a latest news about AI and its impact on society.")

[{'url': 'https://prometai.app/blog/impact-of-artificial-intelligence-on-society-2026',
  'content': '14 June 2026\n\nTable of Contents\n\nAI Business PlanAI for BusinessAI trends\n\nThe Impact of Artificial Intelligence on Society in 2026)\n\nEvery era is defined by the force that changes how people live and think. For the twenty-first century, that force is artificial intelligence. What began as an experiment in machine learning has evolved into a powerful presence shaping nearly every aspect of modern life.\n\nIn 2026, AI influences the jobs people pursue, the way they learn, the care they receive, and even the choices they make online. It drives innovation, fuels economies, and challenges society to balance progress with purpose. The age of intelligent technology is here, and the way humanity responds will define the world that follows.\n\n## The AI Revolution Reshaping Our World [...] Home\n\nThe PrometAI Blog: Business Planning ...\n\nThe Impact of Artificial Intelligence ...\n\n

In [6]:
#============================================
#LLM
#=============================================

from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [7]:
response =llm.invoke("Write a short story about love.")
print(response.content)

Once upon a time, in a small village nestled in the rolling hills of the countryside, there lived a young woman named Lily. She was known throughout the village for her kindness and beauty, but she had yet to find true love.

One day, a handsome stranger arrived in the village. His name was James, and he had traveled from a far-off land in search of adventure. When he laid eyes on Lily, he was immediately captivated by her grace and charm.

As the days passed, James and Lily found themselves drawn to each other. They spent hours talking and laughing, discovering that they shared a deep connection and understanding of each other. It wasn't long before they realized that they had fallen in love.

Their love blossomed like the flowers in the meadow, growing stronger with each passing day. They went on long walks through the fields, hand in hand, and watched the sunset together, lost in each other's eyes.

But as the seasons changed and the leaves began to fall, James knew that he could no

In [33]:
#============================================
#Prompt
#============================================
prompt = hub.pull("hwchase17/react")

/opt/anaconda3/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [34]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [35]:
#============================================
#TOOLS
#============================================
tools = [search_tool, get_weather_data]
#tools = []

In [36]:
#============================================
#CREATE AGENT
#============================================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [37]:
#============================================
#Executor
#============================================
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False
)

In [38]:
#============================================
#Run
#============================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather"
        )
    })

In [21]:
print(response["output"])

The capital of India is New Delhi, which has a rich history dating back to ancient civilizations and has served as the seat of power for various empires and dynasties.
